In [ ]:
# ============================================================
# Test-retest reliability
# ============================================================
#
# Test-retest reliability is assessed in a subset of 36
# participants by correlating ratings from the first and
# second testing sessions.
#
# Pearson correlations are calculated separately for each
# rating dimension and participant.
# ============================================================

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import glob
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t


# ============================================================
# 2. Load datasets
# ============================================================

test_files = sorted(
    glob.glob(
        "../materials/test_retest/T0_test_datasets/*.csv"
    )
)

retest_files = sorted(
    glob.glob(
        "../materials/test_retest/T1_retest_datasets/*.csv"
    )
)

print(f"Test datasets: {len(test_files)}")
print(f"Retest datasets: {len(retest_files)}")


In [ ]:
# ============================================================
# 3. Compute correlations for each participant and dimension
# ============================================================

participant_results = {}

for test_file, retest_file in zip(test_files, retest_files):

    test_df = (pd.read_csv(test_file, sep=",",index_col=0).sort_index())

    retest_df = (pd.read_csv(retest_file,sep=",", index_col=0).sort_index())

    # Retain only common words
    test_meaning = pd.to_numeric(test_df.get("meaning"),errors="coerce")
    retest_meaning = pd.to_numeric(retest_df.get("meaning"),errors="coerce")
    common_words = test_df.index.intersection(retest_df.index)
    keep_words = common_words[
        (test_meaning.reindex(common_words) == 0)
        &
        (retest_meaning.reindex(common_words) == 0)
    ]

    test_df = test_df.loc[keep_words]
    retest_df = retest_df.loc[keep_words]

    # Exclude variables that are not part of the
    # test-retest reliability analysis.
    test_df = test_df.drop(columns=["valence_abs", "meaning"],errors="ignore")
    retest_df = retest_df.drop(columns=["valence_abs", "meaning"],errors="ignore")

    # Align words and dimensions across the two sessions.
    test_aligned, retest_aligned = test_df.align(retest_df,join="inner",axis=0)
    test_aligned, retest_aligned = test_aligned.align(retest_aligned,join="inner",axis=1)

    dimensions = test_aligned.columns

    # Extract participant ID from the filename.
    participant_id = Path(
        test_file
    ).stem.split("_")[0]

    print(participant_id)

    results = []

    for dimension in dimensions:

        test_ratings = pd.to_numeric(
            test_aligned[dimension],
            errors="coerce"
        )

        retest_ratings = pd.to_numeric(
            retest_aligned[dimension],
            errors="coerce"
        )

        valid_pairs = (
            np.isfinite(test_ratings)
            &
            np.isfinite(retest_ratings)
        )

        n_valid_pairs = int(
            valid_pairs.sum()
        )

        if n_valid_pairs < 3:
            correlation = np.nan
            p_value = np.nan

        else:
            correlation, p_value = stats.pearsonr(
                test_ratings[valid_pairs],
                retest_ratings[valid_pairs]
            )

        results.append(
            (
                dimension,
                correlation,
                p_value,
                n_valid_pairs
            )
        )

    participant_correlations = pd.DataFrame(
        results,
        columns=[
            "Dimension",
            "r",
            "p",
            "n_pairs"
        ]
    ).set_index("Dimension")

    participant_results[
        participant_id
    ] = participant_correlations

    print(
        participant_correlations,
        "\n"
    )


In [ ]:
# ============================================================
# 4. Combine participant-level correlations
# ============================================================

participant_correlations_long = (
    pd.concat(
        {participant_id: correlations["r"]
            for participant_id, correlations
            in participant_results.items()
        },
        names=[
            "participant",
            "Dimension"
        ]
    )
    .rename("r")
    .reset_index()
)


# ============================================================
# 5. Summarize test-retest reliability across participants
# ============================================================

def fisher_summary(r_values):
    """
    Summarize participant-level correlations using
    Fisher's r-to-z transformation.
    """

    r = np.asarray(
        r_values,
        dtype=float
    )

    r = r[
        np.isfinite(r)
    ]

    n = len(r)

    # Fisher r-to-z transformation
    r_clipped = np.clip(
        r,
        -0.999999,
        0.999999
    )

    z = np.arctanh(
        r_clipped
    )

    mean_z = np.mean(z)

    mean_r_fisher = np.tanh(
        mean_z
    )

    # 95% confidence interval on the Fisher-z scale
    se_z = (
        np.std(
            z,
            ddof=1
        )
        /
        np.sqrt(n)
    )

    t_critical = t.ppf(
        0.975,
        df=n - 1
    )

    ci_low_z = (
        mean_z
        -
        t_critical * se_z
    )

    ci_high_z = (
        mean_z
        +
        t_critical * se_z
    )

    return pd.Series({
        "mean_r_fisher": mean_r_fisher,
        "ci95_low": np.tanh(ci_low_z),
        "ci95_high": np.tanh(ci_high_z),
    })


reliability_summary = (
    participant_correlations_long
    .groupby("Dimension")["r"]
    .apply(fisher_summary)
    .unstack()
    .sort_values(
        "mean_r_fisher",
        ascending=False
    )
)


# ============================================================
# 6. Display results
# ============================================================

print(
    reliability_summary.round(2)
)